<a href="https://colab.research.google.com/github/ekrombouts/GenCareAI/blob/work_in_progress/scripts/work_in_progress/422_sampc_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install necessary libraries
!pip install -q transformers datasets

In [2]:
# Cell 2: Import required libraries and mount Google Drive
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

In [3]:
# Cell 3: Load pre-trained fine-tuned model and tokenizer from Google Drive
model_finetuned = "ekrombouts/gcai_sampc_fietje"  # Path to your fine-tuned model
model = AutoModelForCausalLM.from_pretrained(model_finetuned, torch_dtype=torch.bfloat16, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(model_finetuned)
tokenizer.pad_token = tokenizer.eos_token  # Set pad token


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
# Cell 4: Load dataset and retrieve sample 6
path_hf_sampc = "ekrombouts/Galaxy_SAMPC_long"
dataset = load_dataset(path_hf_sampc)
val_dataset = dataset['validation']


In [5]:
# Cell 5: Prepare the prompt with truncated notes from sample 6
sample = val_dataset[13]
prompt = sample['prompt']
print(prompt)

Lees de volgende rapportages en beschrijf de cognitie en gedragsproblemen van de cliënt.

Rapportages:
Cliënt was vanmorgen erg onrustig. Extra aandacht en geruststelling geboden om de situatie te kalmeren.
Meneer heeft goed gegeten tijdens de lunch. Hulp was hierbij nodig maar hij was zelf actiever en alert.
Client was vermoeid en wilde rusten in zijn kamer. Verpleegkundige heeft gezorgd voor een rustige omgeving.
Mevrouw had vanmorgen last van incontinentie. Schone kleding aangetrokken en extra aandacht geboden om haar op haar gemak te stellen.
Dhr had het erg warm en wilde extra drinken. Flesje water aangeboden en tevens controle gedaan op zijn bloedsuikerwaarde.
Meneer had vanmiddag begeleiding nodig bij het doen van een dutje. Rustig in slaap gevallen nadat er rustige muziek is opgezet.
Meneer kreeg hulp bij het wassen en aankleden vanochtend. Hij vertoonde wat achterdochtig gedrag tijdens het eten, maar werd gerustgesteld door het verzorgend personeel.
Tijdens de middagactiviteit

In [6]:
# Cell 6: Tokenize the prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
attention_mask = tokenizer(prompt, return_tensors="pt", padding=True).attention_mask.to(model.device)


In [7]:
# Cell 7: Enable cache and set model to evaluation mode
model.config.use_cache = True
model.eval()


PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiSdpaAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (dense): Linear(in_features=2560, out_features=2560, bias=True)
          (rotary_emb): PhiRotaryEmbedding()
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (final_layernorm): LayerNorm((256

In [8]:
# Cell 8: Generate output from the model
output = model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_new_tokens=150,
    do_sample=True,
    top_p=0.95,
    top_k=50,
    temperature=0.7,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)


In [9]:
# Cell 9: Decode the generated text
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
generated_response = generated_text[len(prompt):].strip()


In [10]:
# Cell 10: Display the generated response and actual response
ref_response = sample['reference']  # Reference response from dataset
print("GENERATED RESPONSE:")
print(generated_response)
print("\nREFERENCE RESPONSE:")
print(ref_response)

GENERATED RESPONSE:
['Vergeetachtigheid' 'Achterdochtig gedrag' 'Moeite met concentratie'
 'Vermoeidheid na cognitieve activiteiten']

REFERENCE RESPONSE:
['Moeite met concentratie en vergeetachtigheid'
 'Achterdochtig gedrag tijdens eten'
 'Vermoeidheid tijdens cognitieve activiteiten']


In [11]:
rapportages = """U was inc van urine. U was niet vriendelijk tijdens het verschonen.
Mw was vanmorgen incontinent van dunne def, bed was ook nat. Mw is volledig verzorgd, bed is verschoond,
Mw. haar kledingkast is opgeruimd.
Mw. zei:"oooh kind, ik heb zo'n pijn. Mijn benen. Dat gaat nooit meer weg." Mw. zat in haar rolstoel en haar gezicht trok weg van de pijn en kreeg traanogen. Mw. werkte goed mee tijdens adl. en was vriendelijk aanwezig. Pijn. Mw. kreeg haar medicatie in de ochtend, waaronder pijnstillers. 1 uur later adl. gegeven.
Ik lig hier maar voor Piet Snot. Mw. was klaarwakker tijdens eerste controle. Ze wilde iets, maar wist niet wat. Mw. een slokje water gegeven en uitgelegd hoe ze kon bellen als ze iets wilde. Mw. pakte mijn hand en bedankte me.
Mevr. in de ochtend ondersteund met wassen en aankleden. Mevr was rustig aanwezig.
Mw is volledig geholpen met ochtendzorg, mw haar haren zijn gewassen. Mw haar nagels zijn kort geknipt.
Mevr heeft het ontbijt op bed genuttigd. Daarna mocht ik na de tweede poging Mevr ondersteunen met wassen en aankleden.
Vanmorgen met mw naar buiten geweest om een sigaret te roken. Mw was niet erg spraakzaam en mw kwam op mij over alsof ze geen behoefte had aan een gesprek. Mw kreeg het koud door de wind en wilde snel weer naar binnen.
"""

In [12]:
def make_prompt(text, category):
    if category == "somatiek":
        prompt = f"""Lees de volgende rapportages en beschrijf de lichamelijke klachten van de cliënt.

Rapportages:
{text}

Geef de output als lijst van strings: ['foo', 'bar'].

Beschrijf de lichamelijke klachten:
"""
    elif category == "adl":
        prompt = f"""Lees de volgende rapportages en beschrijf welke hulp de cliënt nodig heeft bij wassen en kleden.

Rapportages:
{text}

Beschrijf de hulp bij wassen en kleden:
"""
    elif category == "continentie":
        prompt = f"""Lees de volgende rapportages en beschrijf de continentie van de cliënt.

Rapportages:
{text}

Beschrijf de continentie van de cliënt:
"""
    elif category == "mobiliteit":
        prompt = f"""Lees de volgende rapportages en beschrijf de mobiliteit van de cliënt.

Rapportages:
{text}

Beschrijf de mobiliteit van de cliënt:
"""
    elif category == "maatschappelijk":
        prompt = f"""Lees de volgende rapportages en beschrijf de bijzonderheden rondom familie en dagbesteding van de cliënt.

Rapportages:
{text}

Beschrijf de familie en dagbesteding:
"""
    elif category == "psychisch":
        prompt = f"""Lees de volgende rapportages en beschrijf de cognitie en gedragsproblemen van de cliënt.

Rapportages:
{text}

Geef de output als lijst van strings: ['foo', 'bar'].

Beschrijf cognitie en gedragsproblemen:
"""
    return prompt

In [31]:
def answer(notes, category=None):
    prompt = make_prompt(notes, category)

    # Tokenize the prompt
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    attention_mask = tokenizer(prompt, return_tensors="pt", padding=True).attention_mask.to(model.device)

    # Generate output from the model
    output = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=150,
        do_sample=True,
        top_p=0.95,
        top_k=50,
        temperature=0.9,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    generated_response = generated_text[len(prompt):].strip()
    return generated_response

In [32]:
answer(rapportages, "somatiek")

"['Pijn in de benen' 'Pijn die niet meer weg gaat']"

In [33]:
answer(rapportages, "adl")

'Mevr is volledig geholpen met wassen, aankleden en haar haren wassen. Ze heeft hulp nodig bij het verschonen van haar kleding.'

In [34]:
answer(rapportages, "continentie")

'Incontinent van urine en dunne def.'

In [35]:
answer(rapportages, "mobiliteit")

'Mw. is rolstoelafhankelijk, maar kan met ondersteuning naar buiten voor een korte wandeling.'

In [36]:
answer(rapportages, "maatschappelijk")

'Mw. had behoefte aan een gesprek maar was niet erg spraakzaam. Er is een sigaret gerookt in de buitenlucht.'

In [37]:
answer(rapportages, "psychisch")

"['Mw. was klaarwakker tijdens de eerste controle en wilde iets maar wist niet wat.'\n 'Mw. kreeg traanogen van pijn en kreeg pijnstillers.'\n 'Mw. was niet erg spraakzaam en leek geen behoefte te hebben aan een gesprek.']"